# IN403 – Unit 3 Lab: Bayesian vs Neural Models for Medical Risk
**Goal:** Compare a Bayesian model and a neural network for skin cancer risk prediction, focusing on accuracy *and* explainability.


## 1) Setup
Install/import required packages.


In [ ]:
# !pip -q install pgmpy torch scikit-learn pandas matplotlib
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import torch
from torch import nn


## 2) Load data


In [ ]:
df = pd.read_csv('skin_cancer_bayes.csv')
display(df.head())
print(df.describe())


## 3) Baseline Bayesian-style model (Naive Bayes)
Use Naive Bayes as a simple probabilistic baseline.


In [ ]:
from sklearn.naive_bayes import GaussianNB
X = df.drop(columns=['cancer'])
y = df['cancer']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)
nb_proba = nb.predict_proba(X_test)[:,1]
print('NB Accuracy:', accuracy_score(y_test, nb_pred))
print('NB ROC-AUC:', roc_auc_score(y_test, nb_proba))
ConfusionMatrixDisplay(confusion_matrix(y_test, nb_pred)).plot(); plt.title('Naive Bayes'); plt.show()


## 4) Neural Network
Scale features, build a small feed-forward net.


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
Xt = torch.tensor(X_train_s, dtype=torch.float32)
yt = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
Xv = torch.tensor(X_test_s, dtype=torch.float32)
yv = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)
class Net(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,16), nn.ReLU(), nn.Linear(16,8), nn.ReLU(), nn.Linear(8,1), nn.Sigmoid())
    def forward(self,x): return self.net(x)
model = Net(Xt.shape[1])
crit = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loss=[]; val_loss=[]
for e in range(20):
    opt.zero_grad(); p=model(Xt); l=crit(p,yt); l.backward(); opt.step(); train_loss.append(l.item())
    with torch.no_grad(): val_loss.append(crit(model(Xv), yv).item())
plt.plot(train_loss,label='train'); plt.plot(val_loss,label='val'); plt.legend(); plt.title('Loss'); plt.show()
with torch.no_grad():
    nn_proba = model(Xv).numpy().ravel(); nn_pred = (nn_proba>=0.5).astype(int)
print('NN Accuracy:', accuracy_score(y_test, nn_pred))
print('NN ROC-AUC:', roc_auc_score(y_test, nn_proba))
ConfusionMatrixDisplay(confusion_matrix(y_test, nn_pred)).plot(); plt.title('Neural Net'); plt.show()


## 5) Comparison Table


In [ ]:
import pandas as pd
cmp = pd.DataFrame([
 {'model':'Naive Bayes','accuracy':accuracy_score(y_test, nb_pred),'roc_auc':roc_auc_score(y_test, nb_proba)},
 {'model':'Neural Net','accuracy':accuracy_score(y_test, nn_pred),'roc_auc':roc_auc_score(y_test, nn_proba)}])
display(cmp)


## 6) Reflection
Answer in markdown: Which model would you deploy in a hospital and why? Consider explainability and patient safety.
